# FFCWS subset: reading, attention, and maternal SES

Exploratory correlations and a simple OLS model. Run from repo root or from `notebooks/` — the next cell resolves paths automatically.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm

sns.set_theme(style="whitegrid")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "child_dev_analysis_ready.csv"
FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

## Load data

Drop rows with missing education so OLS uses complete cases (one row in the committed CSV).

In [ ]:
df = pd.read_csv(DATA_PATH).dropna()
df["log_income"] = np.log1p(df["mother_household_income"])
df.shape

## Summary statistics

In [ ]:
df.describe().round(2)

## Correlations

In [ ]:
cols = [
    "reading_standard_score",
    "focused_attention_score",
    "mother_household_income",
    "mother_education",
]
corr = df[cols].corr()
corr

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
ax.set_title("Pearson correlations")
plt.tight_layout()
fig.savefig(FIG_DIR / "corr_heatmap.png", dpi=150)
plt.show()

## OLS: reading ~ attention + log income + education

Education enters as a numeric code (ordinal simplification). Interpret as association, not causation.

In [ ]:
X = sm.add_constant(df[["focused_attention_score", "log_income", "mother_education"]])
y = df["reading_standard_score"]
model = sm.OLS(y, X).fit()
model.summary()

## Scatter plots

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.scatterplot(
    data=df, x="focused_attention_score", y="reading_standard_score", alpha=0.35, ax=ax
)
ax.set_title("Reading vs focused attention")
ax.set_xlabel("Focused attention score")
ax.set_ylabel("Reading standard score")
plt.tight_layout()
fig.savefig(FIG_DIR / "reading_vs_attention.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.scatterplot(data=df, x="log_income", y="reading_standard_score", alpha=0.35, ax=ax)
ax.set_title("Reading vs log(1 + household income)")
ax.set_xlabel("log(1 + mother household income)")
ax.set_ylabel("Reading standard score")
plt.tight_layout()
fig.savefig(FIG_DIR / "reading_vs_log_income.png", dpi=150)
plt.show()